In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
torch.set_grad_enabled(False)
torch.set_default_dtype(dtype)
torch.set_printoptions(precision=30, sci_mode=False)

print(f"Device: {device}")

In [ ]:
from typing import Literal

ActivationFunction = Literal["sigmoid", "tanh"]

In [ ]:
import numpy as np
from core.extras import rescaled_reasoning
from core.methods import (
    interval_state_space,
    warm_start_symbolic_state_space,
    rectified_intersection_state_space,
    containment_guided_state_space,
    symbolic_state_space,
)
from core.metrics import covering, is_inside_state_series, state_series_from_samples
from core.symbolic import concretize_state_space
from core.utils import generate_input_samples, generate_inputs, generate_random_matrix
from experimental.methods import (
    periodic_reset_state_space,
    iterative_periodic_reset_state_space,
)
from luis.plot import compare_state_evolutions_and_simulations


MAX_ITERS = 50

n_concepts = 12
density = 0.4
activation: ActivationFunction = "sigmoid"  # "tanh"

phi = 0.4
n_samples = 100
l_0 = 0
u_0 = 1
ADAPTIVE_THRESHOLD = 0.3


seed = int(np.random.randint(0, int(1e5)))
# seed = 36850
# seed = 83907
# seed = 9066
# seed = 36201
# seed = 16607

print(f"SEED = {seed}")

np.random.seed(seed)
torch.manual_seed(seed)

W = generate_random_matrix(n_concepts, density, device=device, dtype=dtype)

Wt = W.T
inputs = generate_inputs(n_concepts, l_0, u_0, device=device).to(dtype=dtype)

naive_states = interval_state_space(
    Wt,
    inputs,
    max_iters=MAX_ITERS,
    phi=phi,
    activation_function=activation,
    device=device,
)

A = generate_input_samples(n_samples, n_concepts, l_0, u_0, device=device)


symbolic_adaptive_states = warm_start_symbolic_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    max_iters=MAX_ITERS,
    device=device,
    arithmetic_iters=5,
)

adaptive_states = concretize_state_space(
    symbolic_adaptive_states,
    inputs,
)

simulated_states = rescaled_reasoning(
    A,
    W,
    T=MAX_ITERS,
    phi=phi,
    transfer_function=torch.sigmoid if activation == "sigmoid" else torch.tanh,
    device=device,
)

sampled_states = state_series_from_samples(simulated_states)

iterative_fixed_reset_states = periodic_reset_state_space(
    Wt,
    inputs,
    activation,
    phi=phi,
    t=-1,
    max_iters=MAX_ITERS,
    arithmetic_warming_iters=5,
    baseline_interval_series=naive_states,
    device=device,
)


# Debug: print shapes
print(f"sampled_states shape: {sampled_states.shape}")
print(f"adaptive_states shape: {adaptive_states.shape}")
print(f"naive_states shape: {naive_states.shape}")

print(f"Is inside adaptive: {is_inside_state_series(sampled_states, adaptive_states)}")

print(f"Naive Covering: {covering(naive_states)}")
print(f"Adaptive Covering: {covering(adaptive_states)}")
print(f"Iterative Fixed Reset Covering: {covering(iterative_fixed_reset_states)}")


In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    iterative_fixed_reset_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Iterative Fixed",
)

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    adaptive_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Adaptive",
)

In [ ]:
iterative_fixed = iterative_fixed_reset_states[5, 0, :]
adaptive = adaptive_states[5, 0, :]
print(iterative_fixed)
print(adaptive)

print((iterative_fixed - adaptive) < 1e-9)